In [1]:
# ============================================================
# 1. CHECK GPU
# ============================================================

!nvidia-smi

Sat May 16 14:23:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# ============================================================
# 2. INSTALL REQUIRED PACKAGE
# ============================================================

!pip install -q ultralytics

from ultralytics import YOLO
import os
import shutil
from pathlib import Path

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 64.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
import gdown

In [4]:
!gdown "https://drive.google.com/uc?export=download&id=1XpWWyCfb5Bhdza44vVFs2qOFX8AGDaJP"

Downloading...
From (original): https://drive.google.com/uc?export=download&id=1XpWWyCfb5Bhdza44vVFs2qOFX8AGDaJP
From (redirected): https://drive.google.com/uc?export=download&id=1XpWWyCfb5Bhdza44vVFs2qOFX8AGDaJP&confirm=t&uuid=b2adaafa-c896-4063-b419-806ec2c8a065
To: /content/Spacenet.zip
100% 153M/153M [00:01<00:00, 80.7MB/s]


In [5]:
# ============================================================
# UNZIP DATASET
# ============================================================
# Replace 'dataset.zip' with the uploaded file name if needed
import os
ZIP_FILE = "/content/Spacenet.zip"
EXTRACT_DIR = "/content/dataset"

if os.path.exists(ZIP_FILE):
    !unzip -q "{ZIP_FILE}" -d "{EXTRACT_DIR}"
    print("Dataset extracted to:", EXTRACT_DIR)
else:
    print("Zip file not found. Skip this cell if using Drive.")


Dataset extracted to: /content/dataset


In [6]:
# Set the training labels path:

DATA_YAML_PATH = "/content/dataset/Spacenet/data.yaml"

In [7]:
# ============================================================
# 4. CHOOSE MODEL
# ============================================================
# Participants should choose ONLY from the allowed models.

MODEL_NAME = "yolov8m.pt"   # Example choices: yolov8n.pt, yolov8s.pt, yolov8m.pt

model = YOLO(MODEL_NAME)
print("Selected model:", MODEL_NAME)

Selected model: yolov8m.pt


In [8]:
# ============================================================
# 5. SET HYPERPARAMETERS
# ============================================================
# Participants may tweak these values according to competition rules.

PROJECT_NAME = "yolo_competition"
RUN_NAME = "trial_01"

EPOCHS = 100
IMAGE_SIZE = 768
BATCH_SIZE = 16
OPTIMIZER = "SGD"        # Try: SGD, Adam, AdamW, auto
LR0 = 0.01
LRF = 0.1
MOMENTUM = 0.937
WEIGHT_DECAY = 0.0005
PATIENCE = 20

# Augmentation-related hyperparameters
HSV_H = 0.015
HSV_S = 0.6
HSV_V = 0.4
DEGREES = 5.0
TRANSLATE = 0.1
SCALE = 0.5
SHEAR = 0.0
FLIPLR = 0.5
FLIPUD = 0.0
MOSAIC = 0.7
MIXUP = 0.1

print("Run name:", RUN_NAME)

Run name: trial_01


In [9]:
# ============================================================
# 6. TRAIN THE MODEL
# ============================================================

results = model.train(
    data=DATA_YAML_PATH,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    optimizer=OPTIMIZER,
    lr0=LR0,
    lrf=LRF,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
    patience=PATIENCE,
    hsv_h=HSV_H,
    hsv_s=HSV_S,
    hsv_v=HSV_V,
    degrees=DEGREES,
    translate=TRANSLATE,
    scale=SCALE,
    shear=SHEAR,
    fliplr=FLIPLR,
    flipud=FLIPUD,
    mosaic=MOSAIC,
    mixup=MIXUP,
    project=PROJECT_NAME,
    name=RUN_NAME,
    pretrained=True,
    verbose=True
)

Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/Spacenet/data.yaml, degrees=5.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.6, hsv_v=0.4, imgsz=768, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=0.7, multi_scale=0.0, name=trial_01, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patie

In [10]:
# ============================================================
# 7. VALIDATE THE TRAINED MODEL
# ============================================================

best_model_path = "/content/runs/detect/yolo_competition/trial_01/weights/best.pt"
print("Best model path:", best_model_path)

best_model = YOLO(best_model_path)
metrics = best_model.val(data=DATA_YAML_PATH)

print(metrics)

Best model path: /content/runs/detect/yolo_competition/trial_01/weights/best.pt
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,840,918 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3026.5±331.0 MB/s, size: 175.1 KB)
val: Scanning /content/dataset/Spacenet/valid/labels.cache... 45 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 45/45 14.5Mit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 344, len(boxes) = 2410. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.7s/it 5.2s
                   all         45       2410      0.794      0.786      0.776      0.616
                    cb         25        122 

In [11]:
# ============================================================
# 8. RUN INFERENCE ON SAMPLE IMAGES
# ============================================================
# Replace this path with a valid folder of images

SAMPLE_IMAGE_DIR = "/content/dataset/Spacenet/test/images"

if os.path.exists(SAMPLE_IMAGE_DIR):
    prediction_results = best_model.predict(
        source=SAMPLE_IMAGE_DIR,
        save=True,
        conf=0.25
    )
    print("Inference complete. Prediction images saved.")
else:
    print("Sample image folder not found. Update SAMPLE_IMAGE_DIR before running.")


image 1/16 /content/dataset/Spacenet/test/images/img173_jpg.rf.dfe7c95bd55bf10ad84372ed2e5c5ac7.jpg: 768x768 102 rbs, 47.2ms
image 2/16 /content/dataset/Spacenet/test/images/img174_jpg.rf.c93fd0fa8e58fda373574bb8a281012d.jpg: 768x768 2 cbs, 108 rbs, 40.3ms
image 3/16 /content/dataset/Spacenet/test/images/img238_jpg.rf.a114ba4e2f92a70ed97e882f967f54f6.jpg: 768x768 15 cbs, 23 rbs, 40.4ms
image 4/16 /content/dataset/Spacenet/test/images/img250_jpg.rf.19f425dbdfb9f59f35f8d2a1ee987939.jpg: 768x768 30 rbs, 40.3ms
image 5/16 /content/dataset/Spacenet/test/images/img368_jpg.rf.9056bc7f35675461a802782941302eda.jpg: 768x768 1 cb, 171 rbs, 40.3ms
image 6/16 /content/dataset/Spacenet/test/images/img36_jpg.rf.5e69713044c46e78bc22b76cf89746c5.jpg: 768x768 123 rbs, 40.3ms
image 7/16 /content/dataset/Spacenet/test/images/img470_jpg.rf.fed786addb34c7cf2a876a2de1b913c9.jpg: 768x768 6 cbs, 27 rbs, 40.3ms
image 8/16 /content/dataset/Spacenet/test/images/img528_jpg.rf.50657746668e8355450e446471c494c3.jpg:

In [12]:
# ============================================================
# 9. DISPLAY TRAINING FOLDER CONTENTS
# ============================================================

!find "/content/runs/detect/yolo_competition/trial_014" -maxdepth 3 -type f | sort

find: ‘/content/runs/detect/yolo_competition/trial_014’: No such file or directory


In [13]:
# ============================================================
# 10. COPY BEST MODEL TO A CLEAR LOCATION
# ============================================================

FINAL_MODEL_PATH = "/content/best_submission_model.pt"

if os.path.exists(best_model_path):
    shutil.copy(best_model_path, FINAL_MODEL_PATH)
    print("Copied best model to:", FINAL_MODEL_PATH)
else:
    print("best.pt not found. Make sure training completed successfully.")

Copied best model to: /content/best_submission_model.pt


In [14]:
# ============================================================
# 11. SAVE HYPERPARAMETER RECORD
# ============================================================

hyperparameter_log = f'''
Model: {MODEL_NAME}
Project name: {PROJECT_NAME}
Run name: {RUN_NAME}

epochs: {EPOCHS}
imgsz: {IMAGE_SIZE}
batch: {BATCH_SIZE}
optimizer: {OPTIMIZER}
lr0: {LR0}
lrf: {LRF}
momentum: {MOMENTUM}
weight_decay: {WEIGHT_DECAY}
patience: {PATIENCE}

hsv_h: {HSV_H}
hsv_s: {HSV_S}
hsv_v: {HSV_V}
degrees: {DEGREES}
translate: {TRANSLATE}
scale: {SCALE}
shear: {SHEAR}
fliplr: {FLIPLR}
flipud: {FLIPUD}
mosaic: {MOSAIC}
mixup: {MIXUP}
'''

with open("/content/hyperparameters_used.txt", "w") as f:
    f.write(hyperparameter_log)

print("Saved hyperparameter record to /content/hyperparameters_used.txt")
print(hyperparameter_log)

Saved hyperparameter record to /content/hyperparameters_used.txt

Model: yolov8m.pt
Project name: yolo_competition
Run name: trial_01

epochs: 100
imgsz: 768
batch: 16
optimizer: SGD
lr0: 0.01
lrf: 0.1
momentum: 0.937
weight_decay: 0.0005
patience: 20

hsv_h: 0.015
hsv_s: 0.6
hsv_v: 0.4
degrees: 5.0
translate: 0.1
scale: 0.5
shear: 0.0
fliplr: 0.5
flipud: 0.0
mosaic: 0.7
mixup: 0.1



In [15]:
# ============================================================
# 12. DOWNLOAD SUBMISSION FILES
# ============================================================

from google.colab import files

download_items = ["/content/best_submission_model.pt", "/content/hyperparameters_used.txt"]

for item in download_items:
    if os.path.exists(item):
        files.download(item)
    else:
        print("File not found:", item)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>